In [ ]:
# Semantic Network with Inheritance and Exceptions

knowledge_base = [
    ("animal", "has", "skin"),

    ("fish", "is-a", "animal"),
    ("fish", "can", "swim"),

    ("shark", "is-a", "fish"),

    ("bird", "is-a", "animal"),
    ("bird", "can", "fly"),

    ("penguin", "is-a", "bird"),
    ("penguin", "cannot", "fly"),
]


# Find inheritance chain of a node
def inheritance_chain(node):
    chain = [node]

    while True:
        parent = None

        for subject, relation, obj in knowledge_base:
            if subject == node and relation == "is-a":
                parent = obj
                break

        if parent is None:
            break

        chain.append(parent)
        node = parent

    return chain


# Query the network
def query(node, relation, value):
    chain = inheritance_chain(node)

    for current_node in chain:

        # Check exception first
        for subject, rel, obj in knowledge_base:
            if (subject == current_node and
                    rel == "cannot" and
                    obj == value):
                return "cannot " + value

        # Check local/inherited property
        for subject, rel, obj in knowledge_base:
            if (subject == current_node and
                    rel == relation and
                    obj == value):
                return value

    return "unknown"


# Display knowledge base
print("KNOWLEDGE BASE")

for triple in knowledge_base:
    print(triple)


# Display inheritance chain
print("\nINHERITANCE CHAIN")

for node in ["shark", "penguin"]:
    print(node, "->", " -> ".join(inheritance_chain(node)))


# Queries
print("\nQUERY RESULTS")

queries = [
    ("shark", "has", "skin"),
    ("shark", "can", "swim"),
    ("penguin", "can", "fly"),
    ("bird", "can", "fly"),
    ("penguin", "has", "skin"),
    ("shark", "can", "fly")
]

for node, relation, value in queries:
    result = query(node, relation, value)
    print(f"{node} {relation} {value} -> {result}")

KNOWLEDGE BASE
('animal', 'has', 'skin')
('fish', 'is-a', 'animal')
('fish', 'can', 'swim')
('shark', 'is-a', 'fish')
('bird', 'is-a', 'animal')
('bird', 'can', 'fly')
('penguin', 'is-a', 'bird')
('penguin', 'cannot', 'fly')

INHERITANCE CHAIN
shark -> shark -> fish -> animal
penguin -> penguin -> bird -> animal

QUERY RESULTS
shark has skin -> skin
shark can swim -> swim
penguin can fly -> cannot fly
bird can fly -> fly
penguin has skin -> skin
shark can fly -> unknown


The query penguin can fly returns cannot fly even though the general property of birds is that they can fly. This is because penguin has a specific exception, penguin cannot fly, which overrides the inherited property bird can fly.

Exceptions must be checked before inherited values at every level of the inheritance chain. If the inherited value were checked first, the system could incorrectly conclude that a penguin can fly from the bird node before considering the more specific exception.

# EXERCISE 1

In [ ]:
knowledge_base.extend([
    ("ostrich", "is-a", "bird"),
    ("ostrich", "cannot", "fly"),
    ("ostrich", "can", "run-fast")
])

print("ostrich can fly ->",
      query("ostrich", "can", "fly"))

print("ostrich can run-fast ->",
      query("ostrich", "can", "run-fast"))

print("ostrich has skin ->",
      query("ostrich", "has", "skin"))

ostrich can fly -> cannot fly
ostrich can run-fast -> run-fast
ostrich has skin -> skin


EXERCISE 2

In [ ]:
frames = {
    "animal": {
        "has": "skin"
    },

    "fish": {
        "ako": "animal",
        "can": "swim"
    },

    "shark": {
        "ako": "fish"
    },

    "bird": {
        "ako": "animal",
        "can": "fly"
    },

    "penguin": {
        "ako": "bird",
        "cannot": "fly"
    },

    "ostrich": {
        "ako": "bird",
        "cannot": "fly",
        "can": "run-fast"
    }
}

def frame_query(node, slot, value):
    current = node

    while current is not None:

        frame = frames[current]

        # Exception has priority
        if frame.get("cannot") == value:
            return "cannot " + value

        # Check local slot
        if frame.get(slot) == value:
            return value

        # Move to parent
        current = frame.get("ako")

    return "unknown"


print(frame_query("penguin", "can", "fly"))
print(frame_query("ostrich", "can", "fly"))
print(frame_query("ostrich", "can", "run-fast"))
print(frame_query("shark", "has", "skin"))

cannot fly
cannot fly
run-fast
skin


EXERCISE 3

In [ ]:
knowledge_base = [
    # General knowledge
    ("animal", "has", "skin"),

    # First parent hierarchy
    ("mammal", "is-a", "animal"),
    ("mammal", "has", "hair"),

    # Second parent hierarchy
    ("flying-animal", "is-a", "animal"),
    ("flying-animal", "can", "fly"),

    # Multiple inheritance
    ("bat", "is-a", "mammal"),
    ("bat", "is-a", "flying-animal"),

    # Another example
    ("penguin", "is-a", "bird"),
    ("bird", "is-a", "animal"),
    ("bird", "can", "fly"),
    ("penguin", "cannot", "fly")
]

In [ ]:
from collections import deque


def multiple_parent_query(node, relation, value):

    queue = deque([node])
    visited = set()

    while queue:

        current = queue.popleft()

        # Avoid visiting the same node again
        if current in visited:
            continue

        visited.add(current)

        # --------------------------------
        # 1. Check exception first
        # --------------------------------
        for subject, rel, obj in knowledge_base:
            if (subject == current and
                    rel == "cannot" and
                    obj == value):

                return "cannot " + value

        # --------------------------------
        # 2. Check local property
        # --------------------------------
        for subject, rel, obj in knowledge_base:
            if (subject == current and
                    rel == relation and
                    obj == value):

                return value

        # --------------------------------
        # 3. Add ALL parents to queue
        # --------------------------------
        for subject, rel, obj in knowledge_base:
            if subject == current and rel == "is-a":
                queue.append(obj)

    # Property not found
    return "unknown"


print("bat can fly ->",
      multiple_parent_query("bat", "can", "fly"))

print("bat has hair ->",
      multiple_parent_query("bat", "has", "hair"))

print("bat has skin ->",
      multiple_parent_query("bat", "has", "skin"))

print("bat can swim ->",
      multiple_parent_query("bat", "can", "swim"))

print("penguin can fly ->",
      multiple_parent_query("penguin", "can", "fly"))

bat can fly -> fly
bat has hair -> hair
bat has skin -> skin
bat can swim -> unknown
penguin can fly -> cannot fly


Multiple inheritance allows a concept to inherit properties from more than one parent. In this experiment, Breadth-First Search (BFS) is used to search the inheritance graph because it processes nodes according to their distance from the queried concept. This ensures that more specific concepts are examined before more distant ancestors. When two parents at the same level provide conflicting information, an explicit parent-priority or ordering rule is required to resolve the conflict. This prevents the result from depending on an arbitrary traversal order.